# 🔍 MeteoSwiss POI Finder

Find the **point ID** you need for the meteogram notebook.

MeteoSwiss provides forecasts for ~5600 points across Switzerland — weather stations, towns, and postal code areas. This notebook lets you search by **name**, **PLZ**, or **station abbreviation** to find the right `point_id`.

**How to use:**
1. Run Cell 1 to load the data
2. In Cell 2, change `QUERY` to your search term (e.g. `"Gimel"`, `"1188"`, `"KLO"`)
3. Run Cell 2 — copy the `POI = "..."` line into your meteogram notebook

In [2]:
import httpx
import pandas as pd
from io import StringIO

COLLECTION_ID = "ch.meteoschweiz.ogd-local-forecasting"

META_POINT_URL = (
    f"https://data.geo.admin.ch/{COLLECTION_ID}/"
    "ogd-local-forecasting_meta_point.csv"
)

resp = httpx.get(META_POINT_URL, timeout=30, follow_redirects=True)
resp.raise_for_status()
df_poi = pd.read_csv(StringIO(resp.content.decode("latin-1")), sep=";")

print(f"✓ {len(df_poi)} POIs loaded")

✓ 5629 POIs loaded


In [9]:
# === Enter a PLZ, name, or abbreviation ===
QUERY = "1188"
# ==========================================

q = str(QUERY).strip()
mask = (
    df_poi["station_abbr"].astype(str).str.contains(q, case=False, na=False)
    | df_poi["point_name"].astype(str).str.contains(q, case=False, na=False)
    | (df_poi["postal_code"] == int(q) if q.isdigit() else False)
)
results = df_poi[mask]

if results.empty:
    print(f"No POI found for '{q}'")
elif len(results) == 1:
    row = results.iloc[0]
    print(f"Found 1 POI:\n")
    display(results[["point_id", "station_abbr", "postal_code", "point_name", "point_height_masl"]])
    print(f'\n→ Use in meteogram notebook:')
    print(f'  POI = "{row["point_id"]}"')
else:
    print(f"Found {len(results)} POI(s):\n")
    display(results[["point_id", "station_abbr", "postal_code", "point_name", "point_height_masl"]])
    print(f"\n⚠ Multiple matches — pick the point_id corresponding to your location:")
    for _, row in results.iterrows():
        print(f'  POI = "{row["point_id"]}"   # {row["point_name"]}')

Found 2 POI(s):



,point_id,station_abbr,postal_code,point_name,point_height_masl
1085,118800,NaN,1188.0,Gimel,733.0
1086,118802,NaN,1188.0,St-George,925.0



⚠ Multiple matches — pick the point_id corresponding to your location:
  POI = "118800"   # Gimel
  POI = "118802"   # St-George
